# Whippy_TTS — Colab End-to-End Test

**Before you start**

1. **Runtime → Change runtime type → T4 GPU**
2. Local Whippy API running on port 4000
3. Cloudflare tunnel running (`cloudflared tunnel --url http://localhost:4000`)
4. Colab Secrets (🔑): `WHIPPY_BASE_URL`, `WHIPPY_API_KEY`, `WHIPPY_AGENT_ID`, `WHIPPY_ORGANIZATION_ID`
5. A `reference.wav` clip ready to upload

**Flow:** user text → Whippy Chat API → Chatterbox → `outputs/output.wav` → playback

## GPU verification

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. Switch to Runtime → Change runtime type → GPU."
    )

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)

## Clone repository

In [ ]:
from pathlib import Path
import shutil
import subprocess

REPO_URL = "https://github.com/Ritwik7631/Whippy_TTS.git"
REPO_DIR = Path("/content/Whippy_TTS")
BRANCH = "feature/whippy-integration"

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

!git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

result = subprocess.run(["git", "checkout", BRANCH], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(
        f"Branch '{BRANCH}' not found on GitHub. Push it from your PC first."
    )

print("Working directory:", Path.cwd())
!git log -1 --oneline

## Verify Python version and project files

In [ ]:
import sys
from pathlib import Path

REPO_DIR = Path("/content/Whippy_TTS")
%cd /content/Whippy_TTS

print("Python:", sys.version)
print("Working directory:", Path.cwd())

required_paths = [
    REPO_DIR / "app/whippy_client.py",
    REPO_DIR / "app/config.py",
    REPO_DIR / "config/voice_config.json",
    REPO_DIR / "scripts/test_whippy_to_speech.py",
    REPO_DIR / "scripts/benchmark_whippy_to_speech.py",
    REPO_DIR / ".env.example",
]

missing = [path for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing files:\n" + "\n".join(f"  - {p.relative_to(REPO_DIR)}" for p in missing)
    )

print("All required project files are present.")

## Configure environment variables

Add Colab Secrets in the 🔑 sidebar:

| Secret | Example |
| --- | --- |
| `WHIPPY_BASE_URL` | `https://your-subdomain.trycloudflare.com` |
| `WHIPPY_API_KEY` | Pow session access token |
| `WHIPPY_AGENT_ID` | Agent UUID |
| `WHIPPY_ORGANIZATION_ID` | Organization UUID |

In [ ]:
import os
from getpass import getpass
from pathlib import Path

SECRET_NAMES = [
    "WHIPPY_BASE_URL",
    "WHIPPY_API_KEY",
    "WHIPPY_AGENT_ID",
    "WHIPPY_ORGANIZATION_ID",
]


def load_secret(name: str) -> str:
    try:
        from google.colab import userdata

        value = userdata.get(name).strip()
        if value:
            print(f"Loaded {name} from Colab Secrets")
            return value
    except Exception:
        pass

    prompt = f"Enter {name}: "
    value = getpass(prompt) if name == "WHIPPY_API_KEY" else input(prompt)
    value = value.strip()
    if not value:
        raise ValueError(f"{name} is required")
    return value


env_values = {name: load_secret(name) for name in SECRET_NAMES}
env_values["WHIPPY_BASE_URL"] = env_values["WHIPPY_BASE_URL"].rstrip("/")

for name, value in env_values.items():
    os.environ[name] = value

Path(".env").write_text(
    "\n".join(f"{k}={v}" for k, v in env_values.items()) + "\n",
    encoding="utf-8",
)

print("Wrote .env")
print("WHIPPY_BASE_URL:", env_values["WHIPPY_BASE_URL"])

## Upload `reference.wav`

In [ ]:
from pathlib import Path
from google.colab import files

Path("voices").mkdir(parents=True, exist_ok=True)
target_path = Path("voices/reference.wav")

print("Upload a file named reference.wav")
uploaded = files.upload()

if "reference.wav" not in uploaded:
    raise RuntimeError("Upload a file named reference.wav")

target_path.write_bytes(uploaded["reference.wav"])
print("Saved to:", target_path.resolve())

## Install dependencies

Colab-specific install rules:

- Do **not** run `pip install -r requirements.txt` (breaks torch + pins old setuptools).
- Install `chatterbox-tts` with `--no-deps` so Colab's torch/torchvision stay aligned.
- This cell **restarts the runtime** so imports load cleanly.

In [ ]:
%cd /content/Whippy_TTS

# Keep Colab's torch stack; only add what the notebook needs.
!pip install -q --upgrade setuptools
!pip install -q "chatterbox-tts==0.1.7" --no-deps
!pip install -q "python-dotenv" "resemble-perth==1.0.1"
!pip install -q \
  "conformer" "diffusers" "librosa" "omegaconf" "pykakasi" \
  "pyloudnorm" "s3tokenizer" "safetensors" "spacy-pkuseg" "transformers"

import IPython

print("Dependencies installed. Restarting runtime...")
IPython.Application.instance().kernel.do_shutdown(restart=True)

## After restart

The runtime restarted automatically. **Do not re-run the install cell.**

Continue from **Verify CUDA** below.

## Verify CUDA

In [ ]:
%cd /content/Whippy_TTS

import torch
import torchvision

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required.")

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("CUDA ready on:", torch.cuda.get_device_name(0))

## Sanity check imports

In [ ]:
from chatterbox.tts import ChatterboxTTS

print("chatterbox import OK")

## Load Chatterbox

In [ ]:
%cd /content/Whippy_TTS

import sys
from pathlib import Path

from chatterbox.tts import ChatterboxTTS

ROOT_DIR = Path("/content/Whippy_TTS")
sys.path.insert(0, str(ROOT_DIR))

from app.config import load_voice_config, resolve_repo_path

voice_config = load_voice_config()
reference_audio = resolve_repo_path(voice_config["reference_audio_path"])
generation_config = voice_config["generation"]
output_audio = resolve_repo_path("outputs/output.wav")

if not reference_audio.exists():
    raise FileNotFoundError(reference_audio)

print("Loading Chatterbox model...")
chatterbox_model = ChatterboxTTS.from_pretrained(device="cuda")
print("Chatterbox loaded.")
print("Reference:", reference_audio.resolve())

## Call the Whippy Chat API

In [ ]:
%cd /content/Whippy_TTS

import os
import sys
from pathlib import Path

from dotenv import load_dotenv

ROOT_DIR = Path("/content/Whippy_TTS")
sys.path.insert(0, str(ROOT_DIR))

from app.whippy_client import WhippyApiError, WhippyClient, WhippyConfig

TEST_MESSAGE = "Hello"

load_dotenv(ROOT_DIR / ".env")
whippy_config = WhippyConfig.from_env(os.environ)
client = WhippyClient(whippy_config)

print("Whippy base URL:", whippy_config.base_url)
print("Sending:", TEST_MESSAGE)

try:
    chat_data = client.chat([{"role": "user", "content": TEST_MESSAGE}])
except WhippyApiError as error:
    raise RuntimeError(f"Whippy API failed: {error}") from error

response_text = chat_data.get("response")
if not isinstance(response_text, str) or not response_text.strip():
    raise RuntimeError("Missing data.response from Whippy")

print("\nWhippy agent response:\n")
print(response_text)

## Generate speech

In [ ]:
import time

print("Generating speech...")
generation_started_at = time.perf_counter()

waveform = chatterbox_model.generate(
    response_text,
    audio_prompt_path=str(reference_audio),
    exaggeration=generation_config["exaggeration"],
    cfg_weight=generation_config["cfg_weight"],
)

generation_seconds = time.perf_counter() - generation_started_at
print(f"Generation time: {generation_seconds:.2f}s")

## Save `output.wav`

In [ ]:
import torchaudio as ta

output_audio.parent.mkdir(parents=True, exist_ok=True)
ta.save(str(output_audio), waveform.detach().cpu(), chatterbox_model.sr)
print("Saved to:", output_audio.resolve())

## Play `output.wav`

In [ ]:
from IPython.display import Audio, display

print("Whippy agent response:")
print(response_text)
print()
print(f"TTS generation time: {generation_seconds:.2f}s")
print(f"Output path: {output_audio.resolve()}")
print()
print("Playback:")
display(Audio(filename=str(output_audio), autoplay=False))

## Latency Feasibility Benchmark

Runs `scripts/benchmark_whippy_to_speech.py` against the same recruiter prompt on:

- Original Chatterbox
- Chatterbox Turbo

**Prerequisites:** complete the cells above first (secrets, `reference.wav`, dependency install, CUDA verification).

This benchmark measures Whippy API latency, model loading, TTS generation, audio duration, real-time factor, and total time from user message to saved WAV. Results are provisional engineering signals only — not an automatic go/no-go for live calls.

In [ ]:
%cd /content/Whippy_TTS

from pathlib import Path

benchmark_script = Path("scripts/benchmark_whippy_to_speech.py")
if not benchmark_script.exists():
    raise FileNotFoundError(
        "Missing scripts/benchmark_whippy_to_speech.py in the cloned repo.\n\n"
        "Colab clones from GitHub, not your local PC. Re-run the "
        "'Clone repository' cell after pushing the latest "
        "feature/whippy-integration branch."
    )

!python scripts/benchmark_whippy_to_speech.py

In [ ]:
import json
from pathlib import Path

from IPython.display import HTML, display

results_path = Path("benchmark/results/latest.json")
if not results_path.exists():
    raise FileNotFoundError(
        "Missing benchmark/results/latest.json. Run the benchmark cell first."
    )

results = json.loads(results_path.read_text(encoding="utf-8"))
whippy = results["whippy"]
environment = results["environment"]

rows = []
for model_key in ("original", "turbo"):
    model = results["models"][model_key]
    rows.append(
        {
            "Model": model["name"],
            "Whippy API (s)": whippy["api_latency_seconds"],
            "Model load (s)": model["model_loading_seconds"],
            "TTS gen (s)": model["tts_generation_seconds"],
            "Audio (s)": model["audio_duration_seconds"],
            "RTF": model["real_time_factor"],
            "User msg → WAV (s)": model["total_end_to_end_seconds"],
            "Feasibility": model["feasibility"],
        }
    )

headers = list(rows[0].keys())
table_html = [
    "<h3>Benchmark summary</h3>",
    f"<p><b>GPU:</b> {environment['gpu_name']} "
    f"(device {environment['cuda_device']})</p>",
    f"<p><b>Response:</b> {whippy['response_word_count']} words, "
    f"{whippy['response_character_count']} characters</p>",
    "<table border='1' cellpadding='6' cellspacing='0' style='border-collapse: collapse;'>",
    "<tr>" + "".join(f"<th>{header}</th>" for header in headers) + "</tr>",
]

for row in rows:
    table_html.append(
        "<tr>"
        + "".join(f"<td>{row[header]}</td>" for header in headers)
        + "</tr>"
    )

table_html.append("</table>")
table_html.append(f"<p><i>{results['disclaimer']}</i></p>")

display(HTML("".join(table_html)))
print("Whippy response:\n")
print(whippy["response_text"])

In [ ]:
from IPython.display import Audio, display
from pathlib import Path

original_audio = Path("outputs/benchmark_original.wav")
turbo_audio = Path("outputs/benchmark_turbo.wav")

for label, path in (
    ("Original Chatterbox", original_audio),
    ("Chatterbox Turbo", turbo_audio),
):
    if not path.exists():
        raise FileNotFoundError(path)
    print(label)
    display(Audio(filename=str(path), autoplay=False))
    print()